In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l8.7 Serve via inference API

Finally, put the model behind a request/response handler. `InferenceService` is
transport-agnostic: the same `handle` method works in a test, a queue worker, or a
FastAPI route.

In [ ]:
from pocketlm import train_tiny, pick_config, InferenceService

corpus = open(CORPUS_PATH, encoding="utf-8").read()
model, tok = train_tiny(corpus, pick_config(DEVICE, 1), seed=0)
svc = InferenceService(model, tok)
resp = svc.handle({"prompt": "The ", "max_new_tokens": 20, "seed": 0})
print(resp["completion"])

Wrapping it in FastAPI is a few lines (shown, not run here, to avoid a server in CI):

```python
from fastapi import FastAPI
app = FastAPI()
svc = InferenceService(model, tok)

@app.post("/generate")
def generate_route(req: dict):
    return svc.handle(req)
# uvicorn serves `app`; the handler is exactly what we tested above.
```

That is the whole course end to end: tokenize, build, train, modernize, adapt, and
now serve a decoder-only language model you wrote yourself.

In [ ]:
assert resp["prompt"] == "The "
assert len(resp["completion"]) == len("The ") + 20